# Research notebook — one idea, one Run All

The engine (feature families, `train()`, `infer()`) lives in **`submission.ipynb`**. This
notebook converts it to `_engine.py` and tests ideas against the bank it currently ships,
so every run is also a conversion check.

## How to use

1. Edit **exactly one cell**: the one marked `IDEA`. Define any family classes there and
   list what to test in `IDEAS`.
2. **Run All.**
3. Read the verdict table at the bottom.

Nothing else needs touching. Everything is set up top to bottom: the engine is reconverted,
the base bank is assembled from `submission.ipynb` itself, and the training data is loaded
**only** if some column is not already cached.

## Decision rules

| kind | rule | meaning |
|---|---|---|
| `family`, `params`, `offset` | `delta > 2*se` **and** better on `>= n-1` of the 6 fold-halves | ADOPT |
| `drop` | `delta > -2*se` **and** `delta > -MARGIN` | NON-INFERIOR — safe to drop |

`delta` is the paired TS-AUC difference against the base over 6 fold-halves (3 folds x the
two stop/score halves). `se` is the paired standard error. Anything that does not clear its
rule is a REJECT: put the class in `feature_research/archived_families.py` and move on.

## Where results live

* `cache/research_results.json` — every CV run, keyed by label + a hash of
  (feature names, params, offset, fold seed, seeds, CatBoost on/off). A run whose key is
  already there is reused and printed as `cached`, so re-running costs nothing.
* `cache/idea_log.csv` — one timestamped row per idea per run.
* `cache/new_<family>.npz` — a family's extracted columns, with its `version`. Bump a
  family's `version` (or set `FORCE_EXTRACT = True`) to force re-extraction.

## SMOKE mode

Set `SMOKE = True` in CONFIG and Run All: ~120 synthetic series, tiny models, a separate
`cache_smoke/` directory, no crunch data, no network. It finishes in minutes and exercises
every code path — use it after editing the harness, and before trusting a long real run.

In [1]:
# ===================== CONFIG -- the only settings =====================
SMOKE = False            # True: synthetic mini-dataset, tiny models, separate cache dir -> Run All in minutes, no crunch data
FOLD_SEED = 11
SEEDS = (0, 1, 2)
PATIENCE = 100
WITH_CATBOOST = False   # True: every CV also trains CatBoost (production CAT_PARAMS) and reports the 0.7/0.3 blend
FORCE_EXTRACT = False    # re-extract an idea's family even if cached
MARGIN = 0.0005          # largest loss a drop/prune idea may cost
CACHE = "cache_smoke" if SMOKE else "cache"
RESULTS = f"{CACHE}/research_results.json"
IDEA_LOG = f"{CACHE}/idea_log.csv"

In [2]:
# ===================== ENGINE =====================
# Reconvert submission.ipynb and import it. This is also the conversion check: if the
# submission stops converting, this cell fails and nothing below runs.
# Every module-level import in the notebook belongs here and nowhere else.
import os
import sys
import gc
import json
import time
import math
import csv
import hashlib
import datetime
import subprocess
import importlib
import resource

try:
    import psutil
except ImportError:
    psutil = None

import numpy as np
import pandas as pd
import lightgbm as lgb

subprocess.run([sys.executable, "-m", "crunch_convert", "notebook",
                "submission.ipynb", "_engine.py", "--override"], check=True)

sys.path.insert(0, ".")
import _engine
importlib.reload(_engine)

# `import *` would shadow CONFIG's SEEDS with the engine's production SEEDS (same value
# today, but CONFIG must stay the single place that sets it). Put the config back afterwards.
_CONFIG = {k: globals()[k] for k in
           ("SMOKE", "FOLD_SEED", "SEEDS", "PATIENCE", "WITH_CATBOOST",
            "FORCE_EXTRACT", "MARGIN", "CACHE", "RESULTS", "IDEA_LOG")}
from _engine import *          # REGISTRY, StreamFeatures, build_dataset, ts_auc, LGB_PARAMS, CAT_PARAMS, ...
globals().update(_CONFIG)

# `import *` skips _-prefixed names. _safe and _folded_ts_auc_table are required; the rest
# are conveniences for writing a family in the IDEA cell, and a missing one must not be fatal.
from _engine import _safe, _folded_ts_auc_table

for _priv in ("_CondBase", "_history", "_garch_grid", "_ar_resid", "_SQ2"):
    try:
        globals()[_priv] = getattr(importlib.import_module("_engine"), _priv)
    except (ImportError, AttributeError):
        print(f"note: _engine has no {_priv} (fine unless an idea needs it)")

try:
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
os.makedirs(CACHE, exist_ok=True)

print(f"engine: {len(REGISTRY)} families, {N_FEATURES} features")
print("  " + ", ".join(f"{f.name}({len(f.names())})" for f in REGISTRY))
if WITH_CATBOOST and CatBoostClassifier is None:
    print("WARNING: WITH_CATBOOST is on but catboost is not installed -- blends will be skipped")

warning df5a182d: line 164: column 15: global variable: found potential use of global variable `train_data` that will be commented out
warning df5a182d: line 301: column 11: global variable: found potential use of global variable `crunch_tools` that will be commented out


Tier 1 registered | features so far: 28
research_step / local_test defined -- see the markdown above for the 4-line workflow.
Tier 1 registered | features so far: 28
research_step / local_test defined -- see the markdown above for the 4-line workflow.
engine: 18 families, 147 features
  z2(10), e2(10), tail(6), time(2), lmom(15), wass(3), energy(3), mmd(3), entropy(3), bocpd(3), garch(9), argarch(11), tgarch(11), recres(11), hmm(11), volser(14), dbocpd(5), nullmodel(17)


/Users/balendu/Desktop/structural/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ===================== DATA (lazy) =====================
# Only an extraction needs the raw series. If every column an idea touches is already
# cached, crunch is never loaded and no parquet is read.
_TRAIN_DATA = None


def _smoke_data(n_series=120, seed=7):
    """~120 synthetic series shaped exactly like crunch's (id, x_hist, x_online, tau)."""
    rng = np.random.default_rng(seed)
    out = []
    for i in range(n_series):
        nh = int(rng.integers(300, 900))
        T = int(rng.integers(40, 400))
        xh = rng.standard_t(5, nh).astype(np.float64)
        xo = rng.standard_t(5, T).astype(np.float64)
        tau = None
        if i % 2:                                     # half break: variance step at a random tau
            tau = int(rng.integers(5, T))
            xo[tau:] *= float(rng.uniform(1.5, 2.5))
        out.append((i, xh, xo, tau))
    return out


def get_train_data():
    """The training series, loaded at most once and only when something must be extracted."""
    global _TRAIN_DATA
    if _TRAIN_DATA is None:
        if SMOKE:
            _TRAIN_DATA = _smoke_data()
            print(f"SMOKE: {len(_TRAIN_DATA)} synthetic series")
        else:
            import crunch
            _TRAIN_DATA = list(crunch.load_notebook().load_data()[0])
            print(f"loaded {len(_TRAIN_DATA)} train series")
    return _TRAIN_DATA


print("data: lazy" + (" (SMOKE: synthetic)" if SMOKE else " (crunch, loaded on first extraction)"))

data: lazy (crunch, loaded on first extraction)


In [4]:
# ===================== PRE-BANK FIX -- run AFTER the engine cell and BEFORE the BASE BANK cell =====================
# The engine now ships two families that need FITTED state (dbocpd: priors, nullmodel: network weights). The bank
# cell cannot fit that, so if it does not find their cached columns it tries to extract them and fails with
# "fitted state ... not set". This writes their columns -- from your research runs, which computed exactly these
# features (the null model cross-fitted, as in production) -- to where the bank cell looks: cache/new_<family>.npz,
# tagged with the ENGINE's version so the bank cell accepts them. Run once; afterwards the bank cell just loads them.
import glob, os
import numpy as np

_rows_file = f"{CACHE}/y_all.npy" if os.path.exists(f"{CACHE}/y_all.npy") else f"{CACHE}/X_all.npy"
_n_rows = np.load(_rows_file, mmap_mode="r").shape[0]                # every cached feature array has one row per row
for _fam, _pattern in (("nullmodel", "nullmodel_feats_*.npz"), ("dbocpd", "new_dbocpd.npz")):
    _f = [f for f in REGISTRY if f.name == _fam]
    if not _f:
        print(f"[pre-bank] {_fam}: not in the engine -- nothing to do"); continue
    _want, _ver = list(_f[0].names()), getattr(_f[0], "version", 1)
    _srcs = sorted(glob.glob(f"{CACHE}/{_pattern}"), key=os.path.getmtime)
    if not _srcs:
        print(f"[pre-bank] {_fam}: no research features ({_pattern}) -- run its research cell first"); continue
    with np.load(_srcs[-1]) as _Z:                                     # read fully, then close (may be the same file)
        _have = [str(s) for s in _Z["names"]]
        _miss = [n for n in _want if n not in _have]
        _X = None if _miss else np.ascontiguousarray(_Z["X"][:, [_have.index(n) for n in _want]], dtype=np.float32)
    if _miss:
        print(f"[pre-bank] {_fam}: research file lacks {_miss[:3]} -- cannot reuse"); continue
    assert _X.shape[0] == _n_rows, f"{_fam}: {_X.shape[0]} rows, the cache has {_n_rows}"
    np.savez(f"{CACHE}/new_{_fam}.npz", X=_X, names=np.array(_want), version=_ver)
    print(f"[pre-bank] {_fam}: wrote cache/new_{_fam}.npz {_X.shape} (engine version {_ver})")

[pre-bank] nullmodel: wrote cache/new_nullmodel.npz (5036517, 17) (engine version 1)
[pre-bank] dbocpd: wrote cache/new_dbocpd.npz (5036517, 5) (engine version 1)


In [5]:
# ===================== BASE BANK = whatever submission.ipynb ships =====================
# The base is always exactly active_feature_names(ACTIVE_FAMILIES), in that order -- never a
# hard-coded list, so pruning or adding a family in submission.ipynb is picked up here for free.
SHIP_FAMS = [f.name for f in REGISTRY] if ACTIVE_FAMILIES is None else \
            [f.name for f in REGISTRY if f.name in ACTIVE_FAMILIES]
SHIP = active_feature_names(ACTIVE_FAMILIES)
FAM_NAMES = {f.name: list(f.names()) for f in REGISTRY if f.name in SHIP_FAMS}
FAM_VERSION = {f.name: int(getattr(f, "version", 1)) for f in REGISTRY if f.name in SHIP_FAMS}
_BACKEND = "serial" if SMOKE else None       # SMOKE stays in-process: no loky, no spawn


def hash8(seq):
    return hashlib.sha1("\x00".join(map(str, seq)).encode()).hexdigest()[:8]


# ---- what the cache already holds -------------------------------------------------
_XP = f"{CACHE}/X_all.npy"
X_all, XALL_NAMES = None, []
y_all = t_all = g_all = None
if os.path.exists(_XP) and os.path.exists(f"{CACHE}/names.json"):
    X_all = np.load(_XP, mmap_mode="r")                    # memory-mapped: stays on disk
    XALL_NAMES = json.load(open(f"{CACHE}/names.json"))
    assert X_all.shape[1] == len(XALL_NAMES), "X_all.npy and names.json disagree"
for _k in ("y_all", "t_all", "g_all"):
    _p = f"{CACHE}/{_k}.npy"
    if os.path.exists(_p):
        globals()[_k] = np.load(_p)
if X_all is not None and y_all is not None:
    assert X_all.shape[0] == y_all.size, "X_all.npy and y_all.npy disagree"


def _save_rows(y, t, g):
    """First extraction on a fresh cache defines the row order everything else is checked against."""
    global y_all, t_all, g_all
    y_all, t_all, g_all = y, t, g
    for k, a in (("y_all", y), ("t_all", t), ("g_all", g)):
        np.save(f"{CACHE}/{k}.npy", a)
    print(f"   row index saved: {y.size:,} rows, {np.unique(g).size:,} series")


def _extract_family(fam):
    """Extract one family over every series and cache it with its version."""
    want = FAM_NAMES[fam]
    t0 = time.time()
    Xn, yn, tn, _, gn = build_dataset(get_train_data(), subsample=False,
                                      active_families={fam}, backend=_BACKEND)
    assert Xn.shape[1] == len(want), f"{fam}: emitted {Xn.shape[1]} columns, names() says {len(want)}"
    assert np.isfinite(Xn).all(), f"{fam}: non-finite values in extraction"
    if y_all is None:
        _save_rows(yn, tn, gn)
    else:
        assert np.array_equal(yn, y_all) and np.array_equal(tn, t_all) and np.array_equal(gn, g_all), \
            f"{fam}: extraction row order differs from the cache -- delete {CACHE} and rebuild"
    Xn = np.ascontiguousarray(Xn, dtype=np.float32)
    np.savez(f"{CACHE}/new_{fam}.npz", X=Xn, names=np.array(want),
             version=np.int64(FAM_VERSION[fam]))
    print(f"   {fam}: extracted {Xn.shape} in {(time.time() - t0) / 60:.1f} min")
    return Xn


def _load_family_npz(fam, force=False):
    """The cached array for one family, or None if the cache cannot serve it."""
    path = f"{CACHE}/new_{fam}.npz"
    if force or not os.path.exists(path):
        return None
    with np.load(path) as Z:
        nm = [str(s) for s in Z["names"]]
        ver = int(Z["version"][()]) if "version" in Z else None
        want = FAM_NAMES[fam]
        if [n for n in want if n not in nm]:
            print(f"   {fam}: cached npz lacks shipped columns -- re-extracting")
            return None
        if ver != FAM_VERSION[fam]:
            print(f"   {fam}: cached version {ver} != engine version {FAM_VERSION[fam]} -- re-extracting")
            return None
        X = np.asarray(Z["X"], dtype=np.float32)
    X = np.ascontiguousarray(X[:, [nm.index(n) for n in want]])
    if y_all is not None:
        assert X.shape[0] == y_all.size, f"{fam}: cached npz has {X.shape[0]} rows, cache has {y_all.size}"
    return X


# ---- assemble the bank ------------------------------------------------------------
# Columns taken from X_all.npy carry no version tag (it predates per-family caching), so they
# are trusted by NAME: a family whose columns are all in names.json is read straight from the
# memmap. Bump that family's version AND delete it from names.json if you ever change it.
PROD, PROD_NAMES = [], []          # column indices into the X_all memmap
FAM_ARRAYS = []                    # [(family, float32 array, names)] for everything else
print("bank:")
for _fam in SHIP_FAMS:
    _want = FAM_NAMES[_fam]
    if X_all is not None and not FORCE_EXTRACT and not [n for n in _want if n not in XALL_NAMES]:
        PROD += [XALL_NAMES.index(n) for n in _want]
        PROD_NAMES += _want
        print(f"   {_fam}: {len(_want)} cols from X_all.npy (memmap)")
        continue
    _arr = _load_family_npz(_fam, force=FORCE_EXTRACT)
    if _arr is None:
        _arr = _extract_family(_fam)
    else:
        print(f"   {_fam}: {len(_want)} cols from new_{_fam}.npz")
    FAM_ARRAYS.append((_fam, _arr, _want))
    del _arr
gc.collect()

# Bank column order: memmap columns first, then the per-family arrays. LightGBM is told the
# matching feature_name list, so order is a layout detail; SHIP order is only the base's identity.
BANK_NAMES = PROD_NAMES + [n for _, _, nm in FAM_ARRAYS for n in nm]
assert sorted(BANK_NAMES) == sorted(SHIP), "the assembled bank is not the shipped bank"
assert len(set(BANK_NAMES)) == len(BANK_NAMES), "duplicate column names in the bank"
BASE_LABEL = f"base{len(SHIP)}_{hash8(SHIP)}"

_ram = sum(a.nbytes for _, a, _ in FAM_ARRAYS) / 2 ** 30
print(f"bank ready: {len(BANK_NAMES)} features from {len(SHIP_FAMS)} families | "
      f"{len(PROD)} memmapped + {len(BANK_NAMES) - len(PROD)} in RAM ({_ram:.2f} GB) | {BASE_LABEL}")

# ---- folds (unchanged: 3 folds by series, stop/score halves, stride-subsampled train rows) ----
P2_FOLDS = 3


def _stride_arr(t):
    s = np.full(t.shape, KEEP_EVERY[-1][1], dtype=np.int32)
    for up_to, stride in reversed(KEEP_EVERY):
        s[t < up_to] = stride
    return s


def _sorted_by_t(mask):
    idx = np.flatnonzero(mask)
    return idx[np.argsort(t_all[idx], kind="stable")]


def build_folds(seed=0):
    uniq = np.unique(g_all)
    fold_u = np.empty(uniq.size, dtype=np.int32)
    for k, part in enumerate(np.array_split(np.random.default_rng(seed).permutation(uniq.size), P2_FOLDS)):
        fold_u[part] = k
    fold_row = fold_u[np.searchsorted(uniq, g_all)]
    half_row = (np.random.default_rng(seed + 1).random(uniq.size) < 0.5)[np.searchsorted(uniq, g_all)]
    sub = (t_all % _stride_arr(t_all)) == 0          # rows extract_series(subsample=True) keeps
    folds = []
    for k in range(P2_FOLDS):
        folds.append(dict(tr=_sorted_by_t((fold_row != k) & sub),
                          st=_sorted_by_t((fold_row == k) & half_row),
                          sc=_sorted_by_t((fold_row == k) & ~half_row)))
        print(f"fold {k}: train {folds[-1]['tr'].size:,} | stop {folds[-1]['st'].size:,} | score {folds[-1]['sc'].size:,}")
    return folds


STRIDE_ALL = _stride_arr(t_all)
FOLDS = build_folds(FOLD_SEED)
HALVES = [(fi, swap) for fi in range(len(FOLDS)) for swap in (False, True)]

bank:
   z2: 10 cols from X_all.npy (memmap)
   e2: 10 cols from X_all.npy (memmap)
   tail: 6 cols from X_all.npy (memmap)
   time: 2 cols from X_all.npy (memmap)
   lmom: 15 cols from X_all.npy (memmap)
   wass: 3 cols from X_all.npy (memmap)
   energy: 3 cols from X_all.npy (memmap)
   mmd: 3 cols from X_all.npy (memmap)
   entropy: 3 cols from X_all.npy (memmap)
   bocpd: 3 cols from X_all.npy (memmap)
   garch: 9 cols from new_garch.npz
   argarch: 11 cols from new_argarch.npz
   tgarch: 11 cols from new_tgarch.npz
   recres: 11 cols from new_recres.npz
   hmm: 11 cols from new_hmm.npz
   volser: 14 cols from new_volser.npz
   dbocpd: 5 cols from new_dbocpd.npz
   nullmodel: 17 cols from new_nullmodel.npz
bank ready: 147 features from 18 families | 58 memmapped + 89 in RAM (1.67 GB) | base147_88936ca5
fold 0: train 1,020,112 | stop 848,058 | score 849,407
fold 1: train 1,026,629 | stop 813,776 | score 845,268
fold 2: train 1,022,269 | stop 833,755 | score 846,253


In [6]:
# ===================== HARNESS: one CV engine for every idea =====================
# Every name here is harness-private and used nowhere else; do not reuse them in the IDEA cell.
BUCKETS = ((0, 64), (64, 256), (256, 10 ** 9))
BUCKET_KEYS = [f"t{lo}_{hi if hi < 10 ** 9 else 'inf'}" for lo, hi in BUCKETS]
SMOKE_PARAMS = dict(num_leaves=15, min_data_in_leaf=20, learning_rate=0.1,
                    extra_trees=False, min_gain_to_split=0.0, feature_fraction=0.9)
SMOKE_ROUNDS = 60

RES = json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}


def _save_results():
    with open(RESULTS + ".tmp", "w") as f:
        json.dump(RES, f, indent=1)
    os.replace(RESULTS + ".tmp", RESULTS)


def _lgb_params(params, seed):
    p = dict(LGB_PARAMS)
    if SMOKE:
        p.update(SMOKE_PARAMS)
    p.update(params or {})
    p.update(objective="binary", metric="binary_logloss",
             seed=seed, bagging_seed=seed + 100, feature_fraction_seed=seed + 200)
    return p


def selection(add=(), drop=()):
    """(memmap column indices, [(array, col subset or None)], feature names) for one variant."""
    drop = set(drop)
    prod_idx = [c for c, n in zip(PROD, PROD_NAMES) if n not in drop]
    names = [n for n in PROD_NAMES if n not in drop]
    blocks = []
    for _fam, arr, nm in FAM_ARRAYS:
        keep = [j for j, n in enumerate(nm) if n not in drop]
        if keep:
            blocks.append((arr, None if len(keep) == len(nm) else keep))
            names += [nm[j] for j in keep]
    for arr, nm in add:
        assert arr.shape[0] == y_all.size, f"added block has {arr.shape[0]} rows, expected {y_all.size}"
        assert arr.shape[1] == len(nm), "added block's width does not match its names"
        blocks.append((arr, None))
        names += list(nm)
    assert names, "the selection is empty -- you dropped every column"
    assert len(set(names)) == len(names), "duplicate column names in the selection"
    return prod_idx, blocks, names


def hstack_rows(rows, prod_idx, blocks):
    """The design matrix for `rows`. Name is harness-private on purpose: _engine has its own _stack."""
    parts = []
    if prod_idx:
        parts.append(np.asarray(X_all[rows])[:, prod_idx])
    for arr, cols in blocks:
        a = arr[rows]
        parts.append(a if cols is None else a[:, cols])
    return np.ascontiguousarray(np.hstack(parts) if len(parts) > 1 else parts[0], dtype=np.float32)


def by_bucket(y, t, s):
    out = {"all": float(ts_auc(y, t, s))}
    for (lo, hi), key in zip(BUCKETS, BUCKET_KEYS):
        m = (t >= lo) & (t < hi)
        out[key] = float(ts_auc(y[m], t[m], s[m])) if m.sum() > 100 else float("nan")
    return out


def per_t_offset(t, y, w, nbins=60):
    """Smoothed P(y=1 | t) from the training rows: stride-weighted positive share in log(t)
    bins, linearly interpolated between bin centres. Returns a function t -> logit(rate)."""
    lt = np.log1p(np.asarray(t, dtype=np.float64))
    y = np.asarray(y, dtype=np.float64)
    w = np.asarray(w, dtype=np.float64)
    edges = np.unique(np.quantile(lt, np.linspace(0, 1, nbins + 1)))
    b = np.clip(np.searchsorted(edges, lt, side="right") - 1, 0, max(edges.size - 2, 0))
    n = max(edges.size - 1, 1)
    sw = np.bincount(b, w, minlength=n)
    sy = np.bincount(b, w * y, minlength=n)
    sl = np.bincount(b, w * lt, minlength=n)
    ok = sw > 0
    ctr, rate = sl[ok] / sw[ok], (sy[ok] + 0.5) / (sw[ok] + 1.0)
    rate = np.clip(rate, 1e-4, 1 - 1e-4)
    lg = np.log(rate / (1 - rate))
    return lambda tt: np.interp(np.log1p(np.asarray(tt, dtype=np.float64)), ctr, lg)


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def _run_half(fi, swap, prod_idx, blocks, names, params, offset, seeds):
    """One fold-half: stack the three matrices ONCE, then train every seed on them."""
    F = FOLDS[fi]
    si, ei = (F["sc"], F["st"]) if swap else (F["st"], F["sc"])
    Xtr = hstack_rows(F["tr"], prod_idx, blocks)
    Xs = hstack_rows(si, prod_idx, blocks)
    Xe = hstack_rows(ei, prod_idx, blocks)
    ytr, ys, ye = y_all[F["tr"]], y_all[si], y_all[ei]
    te = t_all[ei]
    w = STRIDE_ALL[F["tr"]].astype(np.float32)
    rounds = SMOKE_ROUNDS if SMOKE else NUM_ROUNDS

    o_tr = o_s = o_e = None
    if offset is not None:
        fn = offset(t_all[F["tr"]], ytr, w)
        o_tr, o_s, o_e = fn(t_all[F["tr"]]), fn(t_all[si]), fn(t_all[ei])

    pe, its = [], []
    for s in seeds:
        dtr = lgb.Dataset(Xtr, label=ytr, weight=w, feature_name=names, init_score=o_tr)
        dva = lgb.Dataset(Xs, label=ys, reference=dtr, init_score=o_s)
        b = lgb.train(_lgb_params(params, s), dtr, num_boost_round=rounds, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(PATIENCE, verbose=False)])
        if offset is None:
            pe.append(b.predict(Xe, num_iteration=b.best_iteration))
        else:   # the booster's output excludes init_score: add it back, then squash
            pe.append(_sigmoid(b.predict(Xe, num_iteration=b.best_iteration, raw_score=True) + o_e))
        its.append(b.best_iteration)
        del dtr, dva, b
        gc.collect()
    p_lgb = np.mean(pe, axis=0)

    out = {"trees": int(np.mean(its))}
    out.update({f"lgb_{k}": v for k, v in by_bucket(ye, te, p_lgb).items()})

    if WITH_CATBOOST and CatBoostClassifier is not None:
        cat = CatBoostClassifier(**CAT_PARAMS, iterations=rounds, early_stopping_rounds=PATIENCE,
                                 random_seed=0, thread_count=8, verbose=0, allow_writing_files=False)
        if offset is None:
            cat.fit(Xtr, ytr, sample_weight=w, eval_set=(Xs, ys))
            p_cat = cat.predict_proba(Xe)[:, 1]
        else:
            from catboost import Pool as _CatPool
            cat.fit(_CatPool(Xtr, label=ytr, weight=w, baseline=o_tr),
                    eval_set=_CatPool(Xs, label=ys, baseline=o_s))
            p_cat = cat.predict(_CatPool(Xe, baseline=o_e), prediction_type="Probability")[:, 1]
        p_bl = (1 - CAT_WEIGHT) * p_lgb + CAT_WEIGHT * p_cat
        out.update({f"blend_{k}": v for k, v in by_bucket(ye, te, p_bl).items()})
        del cat, p_cat, p_bl
    del Xtr, Xs, Xe, pe, p_lgb
    gc.collect()
    return out


def run_cv(label, add=(), drop=(), params=None, offset=None, seeds=SEEDS):
    """6 paired fold-halves on (base - drop + add). Cached in RESULTS; returns the record."""
    prod_idx, blocks, names = selection(add, drop)
    key = hash8([hash8(names), json.dumps(params or {}, sort_keys=True),
                 getattr(offset, "__name__", "none"), FOLD_SEED, tuple(seeds), bool(WITH_CATBOOST)])
    r = RES.get(label)
    if r is not None and r.get("key") == key:
        print(f"[{label}] cached | {r['n_feats']} feats | mean {np.mean(r['lgb_all']):.5f}")
        return r
    r = RES[label] = {"key": key, "n_feats": len(names), "trees": []}
    for fi, swap in HALVES:
        t0 = time.time()
        h = _run_half(fi, swap, prod_idx, blocks, names, params, offset, seeds)
        for k, v in h.items():
            r.setdefault(k, []).append(v)
        _save_results()
        print(f"  [{label}] fold {fi} swap={swap}: {h['lgb_all']:.5f}"
              + (f" | blend {h['blend_all']:.5f}" if "blend_all" in h else "")
              + f" | trees {h['trees']} | {time.time() - t0:.0f}s")
    print(f"[{label}] {r['n_feats']} feats | mean {np.mean(r['lgb_all']):.5f}"
          + (f" | blend {np.mean(r['blend_all']):.5f}" if "blend_all" in r else ""))
    return r


def _paired(d):
    d = np.asarray(d, dtype=np.float64)
    d = d[np.isfinite(d)]
    if d.size < 2:
        return dict(delta=float("nan"), se=float("nan"), sigma=float("nan"), better=0, n=int(d.size))
    m = float(d.mean())
    se = float(d.std(ddof=1) / np.sqrt(d.size))
    return dict(delta=m, se=se, sigma=(m / se if se else 0.0), better=int((d > 0).sum()), n=int(d.size))


def compare(a, b):
    """Paired a - b over the 6 fold-halves: overall, per time bucket, and the blend if present."""
    out = {"all": _paired(np.subtract(a["lgb_all"], b["lgb_all"]))}
    for k in BUCKET_KEYS:
        out[k] = _paired(np.subtract(a[f"lgb_{k}"], b[f"lgb_{k}"]))
    if "blend_all" in a and "blend_all" in b:
        out["blend"] = _paired(np.subtract(a["blend_all"], b["blend_all"]))
    return out


def add_ok(a, b):
    c = compare(a, b)["all"]
    return bool(c["delta"] > 2 * c["se"] and c["better"] >= c["n"] - 1)


def drop_ok(a, b):
    c = compare(a, b)["all"]
    return bool(c["delta"] > -2 * c["se"] and c["delta"] > -MARGIN)


print(f"harness ready | folds seed {FOLD_SEED} | seeds {SEEDS} | patience {PATIENCE} | "
      f"catboost {'on' if WITH_CATBOOST else 'off'} | results -> {RESULTS}"
      + (f"\n  SMOKE: LGB_PARAMS overridden with {SMOKE_PARAMS} and {SMOKE_ROUNDS} rounds" if SMOKE else ""))

harness ready | folds seed 11 | seeds (0, 1, 2) | patience 100 | catboost off | results -> cache/research_results.json


In [7]:
# ===================== IDEA HELPERS =====================
# One entry point per kind of idea. Each returns (label, kind, record, base record, verdict).
def base_column(name):
    """One base column as a 1-D float32 array, from wherever it lives."""
    if name in PROD_NAMES:
        return np.asarray(X_all[:, PROD[PROD_NAMES.index(name)]], dtype=np.float32)
    for _fam, arr, nm in FAM_ARRAYS:
        if name in nm:
            return np.ascontiguousarray(arr[:, nm.index(name)])
    raise KeyError(f"{name} is not in the base bank")


def base_univariate():
    """Univariate TS-AUC of every base column, one column at a time (never the whole matrix).
    Cached per base bank, because it only changes when submission.ipynb's bank changes."""
    path = f"{CACHE}/univariate_{BASE_LABEL}.json"
    if os.path.exists(path):
        return json.load(open(path))
    print(f"   scoring {len(BANK_NAMES)} base columns univariately (once per bank)...")
    rk = TSRanker(t_all)
    tab = {}
    for n in BANK_NAMES:
        col = base_column(n)
        a = float(rk.auc_from_ranks(y_all, rk.ranks(col)))
        tab[n] = max(a, 1 - a)
        del col
    gc.collect()
    json.dump(tab, open(path, "w"), indent=1)
    return tab


def _register(cls):
    """Put `cls` in REGISTRY, replacing any same-named family, and refresh the name/version maps."""
    inst = cls()
    REGISTRY[:] = [f for f in REGISTRY if f.name != inst.name] + [inst]
    FAM_NAMES[inst.name] = list(inst.names())
    FAM_VERSION[inst.name] = int(getattr(inst, "version", 1))
    return inst


def _smoke_family(name, names, n=3):
    """Run the family alone through StreamFeatures on synthetic series: width, finiteness, dead columns."""
    rng = np.random.default_rng(0)
    rows = []
    for k in range(n):
        xh = rng.standard_t(6, int(rng.integers(500, 2500)))
        xo = rng.standard_t(6, 600)
        xo[300:] *= (1.6 if k else 1.0)
        fx = StreamFeatures(xh, active_families={name})
        rows.append(np.array([fx.update(v) for v in xo]))
    R = np.vstack(rows)
    assert R.shape[1] == len(names), f"{name}: emits {R.shape[1]} values, names() says {len(names)}"
    assert np.isfinite(R).all(), f"{name}: non-finite output on synthetic data"
    dead = [names[i] for i in range(R.shape[1]) if R[:, i].std() == 0]
    print("   smoke test OK" + (f" | constant on synthetic data: {dead}" if dead else ""))


def test_family(cls, seeds=SEEDS):
    """Register a family class, extract (or load) it, show its univariate table, CV it against the base."""
    inst = _register(cls)
    name = inst.name
    names = list(inst.names())
    clash = sorted(set(names) & set(BANK_NAMES))
    assert not clash, f"{name}: these column names are already in the base: {clash[:5]}"
    print(f"== {name}: {len(names)} features -> {names}")

    _smoke_family(name, names)

    Xn = _load_family_npz(name, force=FORCE_EXTRACT)
    if Xn is None:
        Xn = _extract_family(name)
    else:
        print(f"   loaded cache new_{name}.npz {Xn.shape}")

    tab = _folded_ts_auc_table(Xn, y_all, t_all, names)
    uni = base_univariate()
    ref = max(uni, key=uni.get)
    rv = base_column(ref).astype(np.float64)
    tab[f"corr_{ref}"] = [abs(np.corrcoef(Xn[:, i].astype(np.float64), rv)[0, 1]) for i in range(len(names))]
    del rv
    gc.collect()
    print(f"   strongest base column: {ref} (univariate TS-AUC {uni[ref]:.5f})")
    print(tab.sort_values("ts_auc_abs", ascending=False).round(4).to_string(index=False))

    base = run_cv(BASE_LABEL, seeds=seeds)
    cand = run_cv(f"{BASE_LABEL}+{name}", add=[(Xn, names)], seeds=seeds)
    del Xn
    gc.collect()                      # the .npz stays on disk; the array does not stay in RAM
    return f"+{name}", "family", cand, base, add_ok(cand, base)


def expand_names(names_or_families):
    """Family names -> their columns; column names pass through. Unknown entries are an error."""
    out = []
    for item in names_or_families:
        if item in FAM_NAMES and item in SHIP_FAMS:
            out += [n for n in FAM_NAMES[item] if n in BANK_NAMES]
        elif item in BANK_NAMES:
            out.append(item)
        else:
            raise KeyError(f"{item!r} is neither a base family nor a base column")
    return out


def test_drop(names_or_families, seeds=SEEDS):
    cols = expand_names(names_or_families)
    label = f"{BASE_LABEL}-{hash8(cols)}"
    print(f"== drop {list(names_or_families)} -> {len(cols)} columns removed")
    base = run_cv(BASE_LABEL, seeds=seeds)
    cand = run_cv(label, drop=cols, seeds=seeds)
    return f"-{','.join(map(str, names_or_families))}", "drop", cand, base, drop_ok(cand, base)


def test_params(label, params, seeds=SEEDS):
    print(f"== params {label}: {params}")
    base = run_cv(BASE_LABEL, seeds=seeds)
    cand = run_cv(f"{BASE_LABEL}#{label}", params=params, seeds=seeds)
    return label, "params", cand, base, add_ok(cand, base)


def test_offset(label, fn=per_t_offset, seeds=SEEDS):
    print(f"== offset {label}: {getattr(fn, '__name__', fn)}")
    base = run_cv(BASE_LABEL, seeds=seeds)
    cand = run_cv(f"{BASE_LABEL}@{label}", offset=fn, seeds=seeds)
    return label, "offset", cand, base, add_ok(cand, base)


def mem(top=8):
    """RSS and the largest globals -- the end-of-run sanity check on a 16 GB laptop."""
    if psutil is not None:
        rss = psutil.Process().memory_info().rss / 2 ** 30
    else:                              # ru_maxrss is bytes on macOS, kilobytes on Linux
        ru = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        rss = ru / 2 ** 30 if sys.platform == "darwin" else ru / 2 ** 20
    big = []
    for k, v in list(globals().items()):
        if k.startswith("__"):
            continue
        if isinstance(v, np.ndarray) and not isinstance(v, np.memmap):
            big.append((v.nbytes, k, f"ndarray{v.shape}"))
        elif isinstance(v, list) and v and isinstance(v[0], tuple):
            nb = sum(a.nbytes for t in v for a in t if isinstance(a, np.ndarray))
            if nb:
                big.append((nb, k, f"list[{len(v)}]"))
    big.sort(reverse=True)
    print(f"RSS {rss:.2f} GB | largest globals: "
          + ", ".join(f"{k} {n / 2 ** 20:.0f} MB {d}" for n, k, d in big[:top]))


print("helpers ready: test_family, test_drop, test_params, test_offset, mem")

helpers ready: test_family, test_drop, test_params, test_offset, mem


### The idea cell

Everything above is setup. Edit the cell below -- define your family classes and fill `IDEAS` -- then Run All and read the verdict table at the bottom.

In [8]:
# ===================== BLIND-SPOT SCAN -- paste after the RUN cell (standalone; IDEAS = [] keeps Run All fast) =====================
# Two questions, both answered for the CURRENT base (with volser):
#
# H  "Is there anything left in the missed breaks?"  For every broken series, ~30 two-sample properties of
#    its whole post-break segment vs its history (location, robust scale, quantiles, tails, shape, serial
#    dependence, volatility clustering, spectrum, periodicity, irreversibility, ordinal patterns, trend,
#    roughness, jumps, vol-of-vol, ties...). The null is the same properties on never-break series, using
#    the last L online points with L matched to the broken series' post-break lengths.
#      H1  per property: how well it separates MISSED / CAUGHT / ALL broken series from the null (AUC of |value|)
#      H2  all properties together: a LightGBM on them, cross-validated -> the hindsight separability of the
#          missed breaks, and which properties it leans on.
#    A property that separates the missed breaks from the null, but that the model does not use, is a blind
#    spot with a name. If nothing separates them even in hindsight, the missed breaks are invisible.
#
# S  "Where is TS-AUC lost?"  An automatic version of the pre-break bucket table: every series descriptor is
#    cut into quartiles; for never-break series (false alarms) and for broken series (misses), each slice's
#    TS-AUC deficit x its share = TS-AUC mass lost there. The top slices are the next targets.
RUN_BLINDSPOT = False
BS_MIN_POST = 64
BS_PATH = f"{CACHE}/blindspot_{BASE_LABEL}_s{FOLD_SEED}.pkl"

if RUN_BLINDSPOT:
    from scipy.special import ndtri as _bs_ndtri
    from scipy.stats import mannwhitneyu as _bs_mwu

    # ---------------- labels -> break time and age ----------------
    _BS_U, _BS_I = np.unique(g_all, return_inverse=True)
    _p = y_all == 1
    BS_TAU = np.full(_BS_U.size, -1, dtype=np.int64)
    _tp = pd.Series(t_all[_p]).groupby(_BS_I[_p]).min()
    BS_TAU[_tp.index.to_numpy()] = _tp.to_numpy()
    BS_AGE = np.where(_p, t_all - BS_TAU[_BS_I], -1)

    # ---------------- out-of-fold predictions of the CURRENT base (cached per base label) ----------------
    def _bs_oof(fi, swap, si, ei):
        p = f"{CACHE}/oof_{BASE_LABEL}_s{FOLD_SEED}_f{fi}{'b' if swap else 'a'}.npz"
        if os.path.exists(p):
            return np.load(p)["p_e"]
        prod_idx, blocks, names = selection()
        F = FOLDS[fi]
        Xtr, Xs, Xe = (hstack_rows(r, prod_idx, blocks) for r in (F["tr"], si, ei))
        w = STRIDE_ALL[F["tr"]].astype(np.float32)
        ps, pe = [], []
        for s in SEEDS:
            dtr = lgb.Dataset(Xtr, label=y_all[F["tr"]], weight=w, feature_name=names)
            dva = lgb.Dataset(Xs, label=y_all[si], reference=dtr)
            b = lgb.train(_lgb_params(None, s), dtr, num_boost_round=SMOKE_ROUNDS if SMOKE else NUM_ROUNDS,
                          valid_sets=[dva], callbacks=[lgb.early_stopping(PATIENCE, verbose=False)])
            ps.append(b.predict(Xs, num_iteration=b.best_iteration))
            pe.append(b.predict(Xe, num_iteration=b.best_iteration))
            del dtr, dva, b
        del Xtr, Xs, Xe
        gc.collect()
        np.savez_compressed(p, p_s=np.mean(ps, axis=0), p_e=np.mean(pe, axis=0))
        print(f"  [oof] fold {fi} swap={swap}: trained + cached")
        return np.mean(pe, axis=0)

    def _bs_pct(t, s, ref):
        """Score percentile among the REF rows at the same t (self excluded)."""
        s = np.clip(np.asarray(s, dtype=np.float64), 0.0, 1.0 - 1e-9)
        tf = t.astype(np.float64)
        key = tf + s
        rk = np.sort(key[ref])
        lo = np.searchsorted(rk, tf, side="left")
        hi = np.searchsorted(rk, tf + 1.0, side="left")
        below = np.searchsorted(rk, key, side="left") - lo
        tie = np.searchsorted(rk, key, side="right") - lo - below
        n = (hi - lo).astype(np.float64) - ref
        return np.where(n > 0, (below + 0.5 * np.maximum(tie - ref, 0)) / np.maximum(n, 1), np.nan)

    frames = []
    for h_, (fi, swap) in enumerate(HALVES):
        F = FOLDS[fi]
        si, ei = (F["sc"], F["st"]) if swap else (F["st"], F["sc"])
        p_e = _bs_oof(fi, swap, si, ei)
        keep = (t_all[ei] % STRIDE_ALL[ei]) == 0
        r = ei[keep]
        k = _BS_I[r]
        yy, tt = y_all[r], t_all[r]
        nb = BS_TAU[k] < 0
        frames.append(pd.DataFrame(dict(half=h_, k=k, y=yy, t=tt, s=p_e[keep], age=BS_AGE[r],
                                        grp=np.where(yy == 1, "pos", np.where(nb, "nobreak", "prebreak")),
                                        pct=_bs_pct(tt, p_e[keep], (yy == 0) & nb))))
    BR = pd.concat(frames, ignore_index=True)
    det = BR[(BR.grp == "pos") & (BR.age >= 32)].groupby("k").pct.mean()           # detection per broken series
    fal = BR[(BR.grp == "nobreak") & (BR.t >= 32)].groupby("k").pct.mean()         # false-alarm level per null series
    print(f"[blindspot] base {BASE_LABEL}: TS-AUC on these rows "
          f"{np.mean([ts_auc(g.y.to_numpy(), g.t.to_numpy(), g.s.to_numpy()) for _, g in BR.groupby('half')]):.4f}")

    # ---------------- hindsight properties (one raw pass, cached) ----------------
    _Z99 = float(_bs_ndtri(0.99))

    def _acf(x, lag):
        x = x - x.mean()
        d = float(x @ x)
        return float(x[lag:] @ x[:-lag]) / d if d > 0 and x.size > lag + 2 else 0.0

    def _skew(x):
        x = x - x.mean()
        s = x.std()
        return float((x ** 3).mean() / s ** 3) if s > 0 else 0.0

    def _kurt(x):
        x = x - x.mean()
        s = x.std()
        return float((x ** 4).mean() / s ** 4 - 3) if s > 0 else 0.0

    def _bands(g, B=64):
        """Welch-style periodogram shares of 4 bands + the peak bin share, on blocks of B."""
        nb_ = g.size // B
        if nb_ < 1:
            return np.full(5, np.nan)
        M = g[:nb_ * B].reshape(nb_, B)
        M = M - M.mean(axis=1, keepdims=True)
        P = (np.abs(np.fft.rfft(M, axis=1)[:, 1:B // 2]) ** 2).mean(axis=0)
        P = P / max(P.sum(), 1e-12)
        return np.array([P[0:4].sum(), P[4:8].sum(), P[8:16].sum(), P[16:].sum(), P.max()])

    def _perm3(g):
        if g.size < 5:
            return np.nan, np.nan
        a, b, c = g[:-2], g[1:-1], g[2:]
        code = (a < b).astype(int) * 4 + (b < c).astype(int) * 2 + (a < c).astype(int)
        cnt = np.bincount(code, minlength=8).astype(float)
        p = cnt[cnt > 0] / cnt.sum()
        mono = (((a < b) & (b < c)) | ((a > b) & (b > c))).mean()
        return float(-(p * np.log(p)).sum()), float(mono)

    def _trend_t(g):
        n = g.size
        if n < 8:
            return 0.0
        x = np.arange(n) - (n - 1) / 2
        b = float(x @ (g - g.mean())) / float(x @ x)
        res = g - g.mean() - b * x
        se = math.sqrt(max(float(res @ res) / (n - 2), 1e-12) / float(x @ x))
        return b / se

    def _volvol(g, w=16):
        nb_ = g.size // w
        if nb_ < 4:
            return np.nan
        v = np.log(g[:nb_ * w].reshape(nb_, w).var(axis=1) + 1e-6)
        return float(v.std())

    def _vr(g, q=8):
        nb_ = g.size // q
        if nb_ < 4:
            return np.nan
        s = g[:nb_ * q].reshape(nb_, q).sum(axis=1)
        return float(s.var() / max(q * g.var(), 1e-12))

    def _jump(g):
        if g.size < 4:
            return np.nan
        bv = (math.pi / 2) * float(np.mean(np.abs(g[1:]) * np.abs(g[:-1])))
        return math.log(max(float(np.mean(g * g)), 1e-12) / max(bv, 1e-12))

    def _one(x, srt, n):
        """All single-segment statistics of x (raw) and its history-PIT Gaussian scores g."""
        u = (np.searchsorted(srt, x, side="left") + 0.5) / (n + 1.0)
        g = _bs_ndtri(np.clip(u, 1e-6, 1 - 1e-6))
        med = float(np.median(x))
        mad = max(float(np.median(np.abs(x - med))) * 1.4826, 1e-12)
        q = np.quantile(g, [0.01, 0.05, 0.25, 0.75, 0.95, 0.99])
        pe, mono = _perm3(g)
        bd = _bands(g)
        ga = np.abs(g)
        dx = np.abs(np.diff(x))
        return dict(med=med, mad=mad, mean=float(x.mean()), var=max(float(x.var()), 1e-12),
                    var_g=max(float(g.var()), 1e-12), q01=q[0], q05=q[1], q25=q[2], q75=q[3], q95=q[4], q99=q[5],
                    tail=float((ga > _Z99).mean()), skew=_skew(g), kurt=_kurt(g),
                    acf1=_acf(g, 1), acf2=_acf(g, 2), acf5=_acf(g, 5), acf10=_acf(g, 10),
                    acfabs1=_acf(ga, 1), acfabs5=_acf(ga, 5),
                    band1=bd[0], band2=bd[1], band3=bd[2], band4=bd[3], peak=bd[4],
                    irrev=_skew(np.diff(g)) if g.size > 3 else 0.0, perm=pe, mono=mono,
                    trend=_trend_t(g), zcross=float((np.diff(np.sign(g)) != 0).mean()) if g.size > 1 else 0.0,
                    ties=float((np.diff(x) == 0).mean()) if x.size > 1 else 0.0,
                    jump=_jump(g), volvol=_volvol(g), vr8=_vr(g), rough=float(dx.mean()) if dx.size else 0.0)

    PROPS = ["med_shift", "mean_shift", "log_mad", "log_var_g", "log_var_raw", "q01", "q05", "q25", "q75", "q95",
             "q99", "tail", "skew", "kurt", "acf1", "acf2", "acf5", "acf10", "acfabs1", "acfabs5", "band1", "band2",
             "band3", "band4", "peak", "irrev", "perm", "mono", "trend", "zcross", "ties", "jump", "volvol", "vr8",
             "rough"]

    def _diff(a, h):
        d = {"med_shift": (a["med"] - h["med"]) / h["mad"], "mean_shift": (a["mean"] - h["mean"]) / math.sqrt(h["var"]),
             "log_mad": math.log(a["mad"] / h["mad"]), "log_var_g": math.log(a["var_g"] / h["var_g"]),
             "log_var_raw": math.log(a["var"] / h["var"]),
             "rough": math.log(max(a["rough"], 1e-12) / max(h["rough"], 1e-12))}
        for c in PROPS:
            if c not in d:
                d[c] = a[c] - h[c]
        return d

    if os.path.exists(BS_PATH):
        HS = pd.read_pickle(BS_PATH)
        print(f"[blindspot] loaded {BS_PATH}")
    else:
        t0 = time.time()
        rng = np.random.default_rng(0)
        raw = {}
        for did, xh, xo, _tau in get_train_data():
            k = int(np.searchsorted(_BS_U, did))
            if k < _BS_U.size and _BS_U[k] == did:
                raw[k] = (xh, xo)
        post_lens = [len(raw[k][1]) - BS_TAU[k] for k in raw if BS_TAU[k] >= 0 and len(raw[k][1]) - BS_TAU[k] >= BS_MIN_POST]
        post_lens = np.asarray(post_lens)
        recs = []
        for k, (xh, xo) in raw.items():
            xh = np.asarray(xh, dtype=np.float64)
            xh = xh[np.isfinite(xh)]
            xo = np.asarray(xo, dtype=np.float64).copy()
            xo[~np.isfinite(xo)] = xh.mean()
            tau = int(BS_TAU[k])
            if tau >= 0:
                seg = xo[tau:]
            else:                                         # null: last L points, L ~ the broken series' post lengths
                ok = post_lens[post_lens <= xo.size]
                seg = xo[-int(rng.choice(ok)):] if ok.size else xo[:0]
            srt, n = np.sort(xh), xh.size
            h = _one(xh, srt, n)
            rec = dict(k=k, broken=tau >= 0, post_len=seg.size, T=xo.size, n_hist=n,
                       # history descriptors for the slice finder
                       h_kurt=h["kurt"], h_skew=h["skew"], h_acf1=h["acf1"], h_acfabs1=h["acfabs1"],
                       h_acfabs5=h["acfabs5"], h_vr8=h["vr8"], h_peak=h["peak"], h_ties=h["ties"],
                       h_trend=h["trend"], h_volvol=h["volvol"], h_jump=h["jump"],
                       h_drift=math.log(max(float(xh[n // 2:].var()), 1e-12) / max(float(xh[:n // 2].var()), 1e-12)))
            if seg.size >= BS_MIN_POST:
                rec.update(_diff(_one(seg, srt, n), h))
            recs.append(rec)
        HS = pd.DataFrame(recs).set_index("k")
        HS.to_pickle(BS_PATH)
        print(f"[blindspot] hindsight properties for {len(HS):,} series | {(time.time() - t0) / 60:.1f} min")

    HS["det"] = det.reindex(HS.index)
    HS["fal"] = fal.reindex(HS.index)
    null = HS[(~HS.broken) & HS.log_mad.notna()]
    pos_ = HS[HS.broken & HS.log_mad.notna() & HS.det.notna()]
    groups = {"MISSED (det<0.6)": pos_[pos_.det < 0.6], "caught (det>=0.8)": pos_[pos_.det >= 0.8], "all broken": pos_}

    def _auc(a, b):
        a, b = a.dropna(), b.dropna()
        if a.size < 20 or b.size < 20:
            return np.nan
        return float(_bs_mwu(a, b).statistic) / (a.size * b.size)

    # ---------------- H1 ----------------
    print(f"\n=== H1: HINDSIGHT SEPARABILITY per property (AUC of |post-break change| vs the same on never-break series;"
          f" 0.5 = no signal) | null n={len(null):,}, " + ", ".join(f"{g} n={len(v):,}" for g, v in groups.items()))
    rows = []
    for c in PROPS:
        rows.append(dict(prop=c, **{g: _auc(v[c].abs(), null[c].abs()) for g, v in groups.items()},
                         missed_median=groups["MISSED (det<0.6)"][c].median(), null_median=null[c].median()))
    H1 = pd.DataFrame(rows).sort_values("MISSED (det<0.6)", ascending=False)
    print(H1.round(3).to_string(index=False))
    print("  (se of each AUC ~0.01-0.02; a property with MISSED well above 0.5 is signal the model is not using)")

    # ---------------- H2 ----------------
    def _cv_auc(Xa, ya, seed=0):
        idx = np.random.default_rng(seed).permutation(len(ya))
        folds = np.array_split(idx, 5)
        oof = np.zeros(len(ya))
        imp = np.zeros(Xa.shape[1])
        for f in folds:
            tr = np.setdiff1d(idx, f)
            m = lgb.train(dict(objective="binary", learning_rate=0.03, num_leaves=15,
                               min_data_in_leaf=int(max(5, min(40, len(tr) // 50))),
                               feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0,
                               verbose=-1, seed=seed, num_threads=8),
                          lgb.Dataset(Xa[tr], label=ya[tr]), num_boost_round=300)
            oof[f] = m.predict(Xa[f])
            imp += m.feature_importance("gain")
        a1 = _auc(pd.Series(oof[ya == 1]), pd.Series(oof[ya == 0]))
        return a1, (imp / imp.sum() if imp.sum() > 0 else imp)

    print("\n=== H2: ALL properties together (5-fold LightGBM on the hindsight properties, series level)")
    for g in ("MISSED (det<0.6)", "all broken"):
        v = groups[g]
        if len(v) < 50:
            print(f"  {g:18s}: too few series ({len(v)})")
            continue
        Xa = np.vstack([v[PROPS].to_numpy(), null[PROPS].to_numpy()]).astype(np.float64)
        ya = np.r_[np.ones(len(v)), np.zeros(len(null))]
        a1, imp = _cv_auc(Xa, ya)
        top = ", ".join(f"{PROPS[i]} {imp[i]:.0%}" for i in np.argsort(imp)[::-1][:8])
        print(f"  {g:18s} vs never-break: hindsight AUC {a1:.3f} | leans on: {top}")
    print("  missed AUC ~0.5 -> the missed breaks are invisible even in hindsight: the edge is NOT on the miss side")
    print("  missed AUC well above 0.5 -> real signal in the missed breaks; build features for the top properties")

    # ---------------- S: slice finder ----------------
    DESC = ["n_hist", "T", "h_kurt", "h_skew", "h_acf1", "h_acfabs1", "h_acfabs5", "h_vr8", "h_peak", "h_ties",
            "h_trend", "h_volvol", "h_jump", "h_drift"]

    def _slice_auc(mask_rows):
        v = []
        for _h, g in BR[mask_rows].groupby("half"):
            if (g.y == 0).sum() > 200 and g.y.sum() > 200:
                v.append(ts_auc(g.y.to_numpy(), g.t.to_numpy(), g.s.to_numpy()))
        return float(np.mean(v)) if v else np.nan

    is_pos, is_nb = BR.grp == "pos", BR.grp == "nobreak"
    base_nb, base_pos = _slice_auc(is_pos | is_nb), _slice_auc(is_pos | (BR.grp != "pos"))
    out = []
    for side in ("false alarms (never-break slice vs all broken)", "misses (broken slice vs all negatives)"):
        for d in DESC + ([] if side.startswith("false") else ["post_len", "log_var_g", "log_mad", "acf1", "acfabs1"]):
            ser = HS[d].abs() if d in ("h_skew", "h_trend", "h_drift", "log_var_g", "log_mad", "acf1", "acfabs1") else HS[d]
            pool = ser[~HS.broken] if side.startswith("false") else ser[HS.broken]
            if pool.notna().sum() < 100:
                continue
            edges = np.r_[-np.inf, pool.dropna().quantile([0.25, 0.5, 0.75]).to_numpy(), np.inf]
            per_row = BR.k.map(ser)
            for qi in range(4):
                inq = (per_row > edges[qi]) & (per_row <= edges[qi + 1])
                if side.startswith("false"):
                    m = is_pos | (is_nb & inq)
                    share = float((is_nb & inq).sum() / max((BR.grp != "pos").sum(), 1))
                    auc_, ref_ = _slice_auc(m), base_nb
                else:
                    m = (is_pos & inq) | (BR.grp != "pos")
                    share = float((is_pos & inq).sum() / max(is_pos.sum(), 1))
                    auc_, ref_ = _slice_auc(m), base_pos
                out.append(dict(side=side.split(" (")[0], descriptor=("|" + d + "|") if d in ("h_skew", "h_trend", "h_drift",
                           "log_var_g", "log_mad", "acf1", "acfabs1") else d, quartile=f"Q{qi + 1}",
                           range=f"{edges[qi]:.3g}..{edges[qi + 1]:.3g}", share=share, tsauc=auc_,
                           deficit=ref_ - auc_, lost_mass=share * (ref_ - auc_)))
    SL = pd.DataFrame(out)
    print(f"\n=== S: WHERE TS-AUC IS LOST (reference: never-break {base_nb:.4f} | all negatives {base_pos:.4f})")
    if SL.empty:
        print("  no slice had enough rows")
    for side in (SL.side.unique() if not SL.empty else []):
        t_ = SL[SL.side == side].sort_values("lost_mass", ascending=False).head(12)
        print(f"  --- {side}: top slices by lost mass (share x deficit)")
        print(t_.drop(columns="side").round(4).to_string(index=False))
    print("  a slice that is both large and deep is the next target; if the same descriptor tops both sides, it is a")
    print("  series TYPE the model handles badly overall (-> per-type calibration or a feature that identifies it)")

In [9]:
RD = pd.read_pickle(f"{CACHE}/step0_raw_desc.pkl")

In [10]:
# ===================== ABRUPTNESS DIAGNOSTIC -- step or drift? (paste after the BLIND-SPOT SCAN cell) =====================
# Real breaks are abrupt (nothing announces them before tau, P2), while never-break series whose volatility wanders
# drift gradually -- and those drifting series are a large, measured false-alarm cost (pair ledger, Step 0).
# No feature measures the SHAPE of a change over time. This measures it, before building anything.
#
# Path = log volatility: log of the 16-step rolling variance of the AR(3) innovations (history coefficients).
# For each series still running at horizon h, on the last W path points ending at the model's step t*
# (for small t* the window reaches back into the history, which gives the pre-break reference):
#   dR2      how much better ONE STEP (best split point) fits than a straight-line TREND   (step - trend)
#   F        how strong the best step is at all (sup-F of step vs no change)
#   jump     size of that step (mean log-vol after - before)          age   log steps since that step
#   trend_t  t-statistic of the linear trend
#   dR2_z, F_z : dR2 and F standardized by the same statistics on 20 random windows of the SERIES' OWN history
#                (the volser idea: what counts as "abrupt" for this series)
# Then, like the horizon gap map: GAP = AUC(model + cell) - AUC(model alone), 5-fold logistic, per horizon, twice --
#   ALL      every broken vs every not-broken series at that step
#   TARGET   broken vs never-break series whose history drifted or whose online segment moved (the costly pairs)
# A clear positive TARGET gap = abruptness separates exactly the pairs you lose -> build the family.
RUN_ABR = False
ABR_HORIZONS = (64, 128, 256, 512)
ABR_W = (128, 512)
ABR_PATH = f"{CACHE}/abrupt_{BASE_LABEL}.pkl"
assert "BR" in globals() and "HS" in globals(), "run the BLIND-SPOT SCAN cell first"

if RUN_ABR:
    from sklearn.linear_model import LogisticRegression as _AbLR

    def _ab_logvol(full, n_h):
        c = full - full[:n_h].mean()
        x = c[:n_h] - c[:n_h].mean()
        d = float(x @ x)
        r = np.array([float(x[k:] @ x[:-k]) / d for k in (1, 2, 3)]) if d > 0 else np.zeros(3)
        R = np.array([[1, r[0], r[1]], [r[0], 1, r[0]], [r[1], r[0], 1]])
        try:
            a = np.linalg.solve(R, r)
        except np.linalg.LinAlgError:
            a = np.zeros(3)
        res = np.zeros_like(c)
        res[3:] = c[3:] - a[0] * c[2:-1] - a[1] * c[1:-2] - a[2] * c[:-3]
        c1, c2 = np.r_[0.0, np.cumsum(res)], np.r_[0.0, np.cumsum(res * res)]
        m = (c1[16:] - c1[:-16]) / 16
        v = np.maximum((c2[16:] - c2[:-16]) / 16 - m * m, 1e-12)
        return 0.5 * np.log(v)                      # lv[i] covers full[i : i+16] -> ends at full index i + 15

    def _ab_stats(p):
        """Step-vs-trend statistics of one window of the log-vol path."""
        W = p.size
        x = np.arange(W, dtype=np.float64)
        sst = float(((p - p.mean()) ** 2).sum()) + 1e-12
        xc = x - x.mean()
        b = float(xc @ (p - p.mean())) / float(xc @ xc)
        sse_tr = float(((p - p.mean() - b * xc) ** 2).sum())
        trend_t = b / math.sqrt(max(sse_tr / (W - 2), 1e-12) / float(xc @ xc))
        c1, c2 = np.cumsum(p), np.cumsum(p * p)
        s = np.arange(8, W - 8, 2)                  # split: [0, s) vs [s, W)
        n1, n2 = s.astype(np.float64), (W - s).astype(np.float64)
        sse1 = c2[s - 1] - c1[s - 1] ** 2 / n1
        sse2 = (c2[-1] - c2[s - 1]) - (c1[-1] - c1[s - 1]) ** 2 / n2
        sse_st = sse1 + sse2
        j = int(np.argmin(sse_st))
        best = float(sse_st[j])
        jump = float((c1[-1] - c1[s[j] - 1]) / n2[j] - c1[s[j] - 1] / n1[j])
        return dict(dR2=(sse_tr - best) / sst, F=(sst - best) / max(best / (W - 2), 1e-12),
                    jump=jump, age=math.log1p(W - s[j]), trend_t=trend_t)

    BRs = BR.sort_values(["k", "t"])
    picks = {}
    for h in ABR_HORIZONS:
        last = BRs[BRs.t <= h - 1].groupby("k").tail(1)
        picks[h] = last[last.t >= h - 1 - 16].set_index("k")[["t", "y", "pct"]]

    if os.path.exists(ABR_PATH):
        AB = pd.read_pickle(ABR_PATH)
        print(f"[abrupt] loaded {ABR_PATH}")
    else:
        t0 = time.time()
        need = {h: set(picks[h].index) for h in ABR_HORIZONS}
        rng = np.random.default_rng(0)
        recs = []
        for did, xh, xo, _tau in get_train_data():
            k = int(np.searchsorted(_BS_U, did))
            if k >= _BS_U.size or _BS_U[k] != did or not any(k in need[h] for h in ABR_HORIZONS):
                continue
            xh = np.asarray(xh, dtype=np.float64)
            xh = xh[np.isfinite(xh)]
            xo = np.asarray(xo, dtype=np.float64).copy()
            xo[~np.isfinite(xo)] = xh.mean()
            lv = _ab_logvol(np.concatenate([xh, xo]), xh.size)
            n_hp = xh.size - 15                         # path points that lie entirely in the history
            cal = {}
            for W in ABR_W:                             # the series' own "normal" for these statistics
                if n_hp >= W + 20:
                    st = [_ab_stats(lv[a:a + W]) for a in rng.integers(0, n_hp - W, 20)]
                    cal[W] = {q: (np.mean([d[q] for d in st]), np.std([d[q] for d in st]) + 1e-9) for q in ("dR2", "F")}
            for h in ABR_HORIZONS:
                if k not in need[h]:
                    continue
                ts = int(picks[h].at[k, "t"])
                end = xh.size + ts - 15                 # path index whose window ends at online step ts
                rec = {"k": k, "h": h}
                for W in ABR_W:
                    if end - W + 1 < 0:
                        continue
                    st = _ab_stats(lv[end - W + 1:end + 1])
                    for q, v in st.items():
                        rec[f"W{W}.{q}"] = v
                    if W in cal:
                        rec[f"W{W}.dR2_z"] = (st["dR2"] - cal[W]["dR2"][0]) / cal[W]["dR2"][1]
                        rec[f"W{W}.F_z"] = (st["F"] - cal[W]["F"][0]) / cal[W]["F"][1]
                recs.append(rec)
        AB = pd.DataFrame(recs)
        AB.to_pickle(ABR_PATH)
        print(f"[abrupt] {len(AB):,} (series, horizon) rows | {(time.time() - t0) / 60:.1f} min")

    _drift_s = (HS.h_drift.abs() >= HS.h_drift.abs()[~HS.broken].quantile(0.75))
    _moved_s = (HS.log_var_g.abs() >= 0.3)
    hard_neg = set(HS.index[(~HS.broken) & (_drift_s | _moved_s)])
    cols = [c for c in AB.columns if "." in c]
    print(f"\n=== ABRUPTNESS | {BASE_LABEL} | GAP = AUC(model + cell) - AUC(model alone)")
    for h in ABR_HORIZONS:
        D = AB[AB.h == h].set_index("k").join(picks[h], how="inner").dropna(subset=["pct"])
        for subset in ("ALL", "TARGET"):
            S = D if subset == "ALL" else D[(D.y == 1) | D.index.isin(hard_neg)]
            y_ = S.y.to_numpy().astype(int)
            if y_.sum() < 100 or (1 - y_).sum() < 100:
                print(f"  h={h} {subset}: too few series"); continue
            m_ = np.log(np.clip(S.pct, 1e-4, 1 - 1e-4) / np.clip(1 - S.pct, 1e-4, 1))
            m_ = m_.to_numpy()
            folds = np.array_split(np.random.default_rng(h).permutation(len(S)), 5)

            def _cv(Xc):
                oof = np.zeros(len(S))
                for f in folds:
                    tr = np.setdiff1d(np.arange(len(S)), f)
                    mu_, sd_ = Xc[tr].mean(0), Xc[tr].std(0) + 1e-9
                    oof[f] = _AbLR(C=1.0, max_iter=500).fit((Xc[tr] - mu_) / sd_, y_[tr]).decision_function((Xc[f] - mu_) / sd_)
                return float(_bs_mwu(oof[y_ == 1], oof[y_ == 0]).statistic) / ((y_ == 1).sum() * (y_ == 0).sum())

            base = _cv(m_[:, None])
            rng_ = np.random.default_rng(7 + h)
            noise = [(lambda z_: _cv(np.column_stack([m_, z_, np.abs(z_)])))(rng_.standard_normal(len(S))) - base
                     for _ in range(4)]
            gaps = {}
            for c in cols:
                v = np.nan_to_num(S[c].to_numpy(dtype=np.float64))
                gaps[c] = _cv(np.column_stack([m_, v, np.abs(v)])) - base
            allg = _cv(np.column_stack([m_, np.nan_to_num(S[cols].to_numpy(dtype=np.float64))])) - base
            top = sorted(gaps.items(), key=lambda kv: -kv[1])[:5]
            print(f"  h={h:3d} {subset:6s}: {len(S):,} series ({y_.sum():,} broken) | model {base:.4f} | all cells {allg:+.4f} | "
                  f"noise {min(noise):+.4f}..{max(noise):+.4f} | top " + ", ".join(f"{c} {v:+.4f}" for c, v in top))
    print("\n  TARGET gaps well above the noise band = the step-vs-drift shape separates the costly pairs -> build it.")

AssertionError: run the BLIND-SPOT SCAN cell first

In [ ]:
# ===================== ABRUPTNESS, FOLLOW-UP -- is the TARGET gain real or a selection effect? =====================
# Paste after the ABRUPTNESS cell (uses its cached statistics AB, picks and cells). Runs in about a minute.
#
# The TARGET set used two definitions of "hard" never-break series:
#   (a) the HISTORY drifted          -> known at the start of the series: a legitimate selection
#   (b) the ONLINE segment moved     -> measured on the whole online segment, INCLUDING steps after t*: hindsight
# Selecting negatives by (b) makes any volatility-trend statistic look informative inside that set, whatever its real
# value, so the big TARGET gaps may be partly manufactured by the selection. Two checks:
#   HIST_ONLY : the target set with (a) only -- no hindsight
#   ALL+INT   : the whole population, with each cell ALSO multiplied by the history-drift level, so the linear model can
#               say "a volatility trend means drift for series that drift, a break for series that do not" -- the
#               kind of interaction the trees would learn if the family were built
_hd = HS.h_drift.abs()
_drift_hist = set(HS.index[(~HS.broken) & (_hd >= _hd[~HS.broken].quantile(0.75))])
print(f"=== ABRUPTNESS FOLLOW-UP | {BASE_LABEL}")
for h in ABR_HORIZONS:
    D = AB[AB.h == h].set_index("k").join(picks[h], how="inner").dropna(subset=["pct"])
    D["hdrift"] = _hd.reindex(D.index).fillna(_hd.median()).to_numpy()
    for subset in ("HIST_ONLY", "ALL+INT"):
        S = D[(D.y == 1) | D.index.isin(_drift_hist)] if subset == "HIST_ONLY" else D
        y_ = S.y.to_numpy().astype(int)
        if y_.sum() < 100 or (1 - y_).sum() < 100:
            print(f"  h={h} {subset}: too few series"); continue
        m_ = np.log(np.clip(S.pct, 1e-4, 1 - 1e-4) / np.clip(1 - S.pct, 1e-4, 1)).to_numpy()
        hd = S.hdrift.to_numpy()
        folds = np.array_split(np.random.default_rng(h).permutation(len(S)), 5)

        def _cv(Xc):
            oof = np.zeros(len(S))
            for f in folds:
                tr = np.setdiff1d(np.arange(len(S)), f)
                mu_, sd_ = Xc[tr].mean(0), Xc[tr].std(0) + 1e-9
                oof[f] = _AbLR(C=1.0, max_iter=500).fit((Xc[tr] - mu_) / sd_, y_[tr]).decision_function((Xc[f] - mu_) / sd_)
            return float(_bs_mwu(oof[y_ == 1], oof[y_ == 0]).statistic) / ((y_ == 1).sum() * (y_ == 0).sum())

        base_X = np.column_stack([m_, hd, m_ * hd]) if subset == "ALL+INT" else m_[:, None]
        base = _cv(base_X)
        rng_ = np.random.default_rng(70 + h)
        noise = []
        for _ in range(4):
            z_ = rng_.standard_normal(len(S))
            extra = np.column_stack([z_, np.abs(z_)] + ([z_ * hd] if subset == "ALL+INT" else []))
            noise.append(_cv(np.column_stack([base_X, extra])) - base)
        gaps = {}
        for c in cols:
            v = np.nan_to_num(S[c].to_numpy(dtype=np.float64))
            extra = np.column_stack([v, np.abs(v)] + ([v * hd] if subset == "ALL+INT" else []))
            gaps[c] = _cv(np.column_stack([base_X, extra])) - base
        V = np.nan_to_num(S[cols].to_numpy(dtype=np.float64))
        allX = np.column_stack([base_X, V] + ([V * hd[:, None]] if subset == "ALL+INT" else []))
        allg = _cv(allX) - base
        top = sorted(gaps.items(), key=lambda kv: -kv[1])[:4]
        print(f"  h={h:3d} {subset:9s}: {len(S):,} series ({y_.sum():,} broken) | model {base:.4f} | all cells {allg:+.4f} | "
              f"noise {min(noise):+.4f}..{max(noise):+.4f} | top " + ", ".join(f"{c} {v:+.4f}" for c, v in top))
print("\n  HIST_ONLY still well above noise AND ALL+INT above noise -> real signal, build the family (with a history-drift")
print("  descriptor next to it so the trees can form the interaction). Both near noise -> the TARGET gain was selection.")

=== ABRUPTNESS FOLLOW-UP | base125_11dda682
  h= 64 HIST_ONLY: 2,188 series (988 broken) | model 0.5336 | all cells +0.1020 | noise -0.0011..+0.0059 | top W512.trend_t +0.0576, W512.F +0.0414, W512.jump +0.0345, W512.dR2 +0.0238
  h= 64 ALL+INT  : 9,588 series (988 broken) | model 0.5627 | all cells -0.0097 | noise -0.0030..+0.0024 | top W512.trend_t +0.0018, W128.age +0.0017, W128.dR2 -0.0002, W128.dR2_z -0.0006
  h=128 HIST_ONLY: 2,549 series (1,418 broken) | model 0.5866 | all cells +0.0554 | noise -0.0017..+0.0006 | top W512.trend_t +0.0237, W512.jump +0.0164, W512.dR2 +0.0119, W512.F +0.0095
  h=128 ALL+INT  : 8,938 series (1,418 broken) | model 0.5958 | all cells +0.0006 | noise -0.0034..+0.0001 | top W128.jump +0.0042, W512.age +0.0019, W512.jump +0.0018, W512.F_z +0.0018
  h=256 HIST_ONLY: 2,848 series (1,895 broken) | model 0.6061 | all cells +0.0351 | noise -0.0035..+0.0011 | top W512.jump +0.0114, W512.F_z +0.0087, W512.trend_t +0.0079, W512.dR2 +0.0057
  h=256 ALL+INT  : 7,

In [11]:
# ===================== GRU STREAM CACHE + dbocpd ON THE GRU'S SURPRISE STREAM =====================
# Three cells. Paste all three after the BLIND-SPOT SCAN cell. WITH_CATBOOST = False. Needs torch.
#
# CELL 0 (run once after shipping dbocpd / nullmodel to submission.ipynb): the research notebook's base now follows
#   the 147-feature bank. Its two new families need fitted state, which the bank cell cannot provide, so their
#   cached columns are written where the bank cell looks (cache/new_<family>.npz, with the engine's version).
# CELL 1 -- the GRU STREAM CACHE you asked for: the three cross-fitted fold models (trained once by the null-model
#   cell, saved as cache/nullmodel_fold{0,1,2}_*.pt) are re-run ONCE and every per-step stream is saved:
#     rows_nll, rows_g  : the online streams, aligned with the cache rows (one value per row of y_all / t_all)
#     hist_nll, hist_g  : each series' history streams (the calibration part), concatenated, with offsets
#   -> cache/gru_streams_s{FOLD_SEED}.npz. Any future idea on the GRU's output starts from this file: no GRU, no torch.
# CELL 2 -- idea #1: the dataset-fitted Bayesian change detector (fast dbocpd recursion) run on the GRU's conditional
#   normal score g (standardized by the series' own history g) instead of on e. Priors refitted on the g scale.
#   5 features, paired CV against the current base.

# ---------------------------------------------------------------- CELL 0: compatibility (run once)
import glob

def _engine_family(name):
    hit = [f for f in REGISTRY if f.name == name]
    return hit[0] if hit else None

for _fam, _src, _prefix in (("nullmodel", "nullmodel_feats_*.npz", "nm_"), ("dbocpd", "new_dbocpd.npz", "dbocpd_")):
    f_ = _engine_family(_fam)
    if f_ is None:
        print(f"[compat] {_fam}: not in the engine -- nothing to do"); continue
    want = list(f_.names())
    dst = f"{CACHE}/new_{_fam}.npz"
    srcs = sorted(glob.glob(f"{CACHE}/{_src}"), key=os.path.getmtime)
    if not srcs:
        print(f"[compat] {_fam}: no research features found ({_src}) -- run its research cell first"); continue
    with np.load(srcs[-1]) as Z:                                  # read fully, then close (dst may be the same file)
        have = [str(s) for s in Z["names"]]
        missing = [n for n in want if n not in have]
        X = None if missing else np.ascontiguousarray(Z["X"][:, [have.index(n) for n in want]], dtype=np.float32)
    if missing:
        print(f"[compat] {_fam}: research file lacks {missing[:3]} -- cannot reuse"); continue
    assert X.shape[0] == y_all.size
    np.savez(dst, X=X, names=np.array(want), version=getattr(f_, "version", 1))
    print(f"[compat] {_fam}: wrote {dst} {X.shape} (engine version {getattr(f_, 'version', 1)})")
print("[compat] now re-run the bank cell so the base picks these up")


# ---------------------------------------------------------------- CELL 1: GRU stream cache
import torch
import torch.nn as nn
from scipy.special import ndtri as _gs_ndtri

GS_PATH = f"{CACHE}/gru_streams_s{FOLD_SEED}.npz"
GS_H, GS_LAYERS, GS_K, GS_WARM, GS_BURN = 64, 2, 5, 1024, 32      # must match the null-model cell
# Run the network on the Mac GPU (MPS) like the null-model cell did. On macOS, PyTorch's CPU thread pool and
# LightGBM's OpenMP runtime can deadlock in the same process (the symptom: 0% CPU, nothing printed); if the CPU is
# used anyway, a single torch thread avoids that.
GS_DEV = "mps" if torch.backends.mps.is_available() else "cpu"
if GS_DEV == "cpu":
    torch.set_num_threads(1)


class _GSNullGRU(nn.Module):                                      # identical architecture to _NullGRU
    def __init__(self):
        super().__init__()
        self.inp = nn.Linear(3, GS_H)
        self.gru = nn.GRU(GS_H, GS_H, num_layers=GS_LAYERS, batch_first=True)
        self.out = nn.Linear(GS_H, 3 * GS_K)

    def forward(self, z):
        x = torch.stack([z, z.abs(), z * z], dim=-1)
        h, _ = self.gru(torch.relu(self.inp(x)))
        lg, mu, ls = self.out(h).chunk(3, dim=-1)
        return torch.log_softmax(lg, dim=-1), mu, ls.clamp(-4.0, 2.0)


if os.path.exists(GS_PATH):
    _G = np.load(GS_PATH)
    GS = {k: _G[k] for k in _G.files}
    print(f"[gru cache] loaded {GS_PATH}: {GS['rows_g'].size:,} online rows, {GS['hist_off'].size - 1:,} series")
else:
    t0 = time.time()
    GU, GI = np.unique(g_all, return_inverse=True)
    fold_of = np.full(GU.size, -1)
    for f, F in enumerate(FOLDS):
        fold_of[np.unique(GI[np.r_[F["st"], F["sc"]]])] = f
    models = []
    for f in range(len(FOLDS)):
        cands = sorted(glob.glob(f"{CACHE}/nullmodel_fold{f}_*.pt"), key=os.path.getmtime)
        assert cands, f"no saved fold-{f} null model -- run the null-model cell once to train it"
        m = _GSNullGRU()
        m.load_state_dict(torch.load(cands[-1], map_location="cpu"))
        m.to(GS_DEV).eval()
        models.append(m)
    ZH, ZO = {}, {}
    for did, xh, xo, _tau in get_train_data():
        k = int(np.searchsorted(GU, did))
        if k >= GU.size or GU[k] != did:
            continue
        xh = np.asarray(xh, dtype=np.float64)
        xh = xh[np.isfinite(xh)]
        xo = np.asarray(xo, dtype=np.float64).copy()
        xo[~np.isfinite(xo)] = xh.mean()
        srt, n = np.sort(xh), xh.size
        zf = lambda v: _gs_ndtri((np.searchsorted(srt, v, side="left") + 0.5) / (n + 1.0)).astype(np.float32)
        ZH[k], ZO[k] = zf(xh), zf(xo)
    rows_nll = np.zeros(y_all.size, dtype=np.float32)
    rows_g = np.zeros(y_all.size, dtype=np.float32)
    hist_parts_nll, hist_parts_g = [None] * GU.size, [None] * GU.size
    print(f"[gru cache] {len(ZH):,} series in z form | streaming on {GS_DEV}")
    for f, model in enumerate(models):
        own = sorted([k for k in ZH if fold_of[k] == f], key=lambda k: min(ZH[k].size, GS_WARM) + ZO[k].size)
        for a in range(0, len(own), 64):
            ks = own[a:a + 64]
            seqs = [np.r_[ZH[k][-GS_WARM:], ZO[k]] for k in ks]
            T = max(s.size for s in seqs)
            arr = np.zeros((len(ks), T), dtype=np.float32)
            for i, s in enumerate(seqs):
                arr[i, :s.size] = s
            xb = torch.from_numpy(arr).to(GS_DEV)
            with torch.no_grad():
                lw, mu, ls = model(xb[:, :-1])
                y = xb[:, 1:].unsqueeze(-1)
                lp = -0.5 * ((y - mu) / ls.exp()) ** 2 - ls - 0.5 * math.log(2 * math.pi)
                nll = (-torch.logsumexp(lw + lp, dim=-1)).cpu().numpy()
                cdf = (lw.exp() * 0.5 * (1 + torch.erf((y - mu) / (ls.exp() * math.sqrt(2))))).sum(-1)
                g = _gs_ndtri(cdf.clamp(1e-6, 1 - 1e-6).cpu().numpy())
            for i, k in enumerate(ks):
                wh, n_on = min(ZH[k].size, GS_WARM), ZO[k].size
                rows = np.flatnonzero(GI == k)
                assert rows.size == n_on and np.array_equal(t_all[rows], np.arange(n_on))
                rows_nll[rows] = nll[i, wh - 1:wh - 1 + n_on]
                rows_g[rows] = g[i, wh - 1:wh - 1 + n_on]
                hist_parts_nll[k] = nll[i, GS_BURN:wh - 1].astype(np.float32)
                hist_parts_g[k] = g[i, GS_BURN:wh - 1].astype(np.float32)
        print(f"  [gru cache] fold {f}: {len(own):,} series streamed | {(time.time() - t0) / 60:.1f} min")
    lens = np.array([0 if p is None else p.size for p in hist_parts_g])
    GS = dict(rows_nll=rows_nll, rows_g=rows_g,
              hist_nll=np.concatenate([p for p in hist_parts_nll if p is not None]),
              hist_g=np.concatenate([p for p in hist_parts_g if p is not None]),
              hist_off=np.r_[0, np.cumsum(lens)], series_ids=GU, fold_of=fold_of,
              meta=np.array(json.dumps(dict(fold_seed=FOLD_SEED, warm=GS_WARM, burn=GS_BURN, H=GS_H, K=GS_K))))
    np.savez(GS_PATH, **GS)
    print(f"[gru cache] saved {GS_PATH} | {(time.time() - t0) / 60:.1f} min")


def gru_history(k, stream="g"):
    """Series k's history stream (the calibration part) from the cache."""
    a, b = GS["hist_off"][k], GS["hist_off"][k + 1]
    return GS[f"hist_{stream}"][a:b]


# ---------------------------------------------------------------- CELL 2: dbocpd on the GRU's g stream
GB_GRID_R = (0.35, 0.5, 0.65, 0.8, 0.9, 1.1, 1.25, 1.5, 1.8, 2.2, 2.8, 3.6)
GB_PATH = f"{CACHE}/gbocpd_s{FOLD_SEED}.npz"
GB_NAMES = ["gbocpd_logodds", "gbocpd_logodds_up", "gbocpd_logodds_down", "gbocpd_age_log", "gbocpd_logratio"]

_GU = GS["series_ids"]
_GI = np.searchsorted(_GU, g_all)
_order = np.lexsort((t_all, _GI))                                  # rows grouped by series, t ascending
_starts = np.flatnonzero(np.r_[True, _GI[_order][1:] != _GI[_order][:-1]])
_ends = np.r_[_starts[1:], _order.size]

# priors, on the g scale: hazard from the online lengths, break sizes from post-break mean(g_std^2)
_tau = pd.Series(t_all[y_all == 1]).groupby(_GI[y_all == 1]).min()
_N = np.bincount(_GI)
_pi = _tau.size / _GU.size
GB_Q = np.array([_pi * float(np.mean(1.0 / _N[_N > t])) if (_N > t).any() else 0.0 for t in range(1000)])
GB_Q = np.minimum(GB_Q, 0.999 / np.arange(1, 1001))
_lr = []
for k, tau in _tau.items():
    h = gru_history(k)
    if h.size < 20:
        continue
    mu_h, sd_h = float(h.mean()), float(h.std()) + 1e-6
    rows = _order[_starts[k]:_ends[k]]
    post = (GS["rows_g"][rows][tau:] - mu_h) / sd_h
    if post.size >= 32:
        _lr.append(math.log(max(float(np.mean(post ** 2)), 1e-6)))
_gl = np.log(np.asarray(GB_GRID_R))
_cnt = np.bincount(np.argmin(np.abs(np.asarray(_lr)[:, None] - _gl[None, :]), axis=1), minlength=_gl.size) + 1.0
GB_LOGW = np.log(_cnt / _cnt.sum())
print("[gbocpd] break-size prior on the g scale: " + ", ".join(f"{r}: {w:.2f}" for r, w in zip(GB_GRID_R, np.exp(GB_LOGW))))


def _gbocpd_series(v):
    """The fast dbocpd recursion over one standardized stream; returns (len(v), 5)."""
    R = _gl.size
    C, logA, logD = np.zeros(R), np.full(R, -np.inf), np.full(R, -np.inf)
    inv_r, half = np.exp(-_gl), 0.5 * _gl
    up = _gl > 0
    out = np.empty((v.size, 5))
    for t, x in enumerate(v):
        x2 = float(x) * float(x)
        logD = np.logaddexp(logD, logA)
        logA = np.logaddexp(logA, -C)
        C = C + (-half - 0.5 * x2 * inv_r + 0.5 * x2)
        E = GB_LOGW + C + logA
        m = float(E.max())
        pr = np.exp(E - m)
        tot = m + math.log(float(pr.sum()))
        q = float(GB_Q[min(t, 999)])
        lq, ln = math.log(max(q, 1e-300)), math.log(max(1.0 - (t + 1) * q, 1e-12))
        if t > 0:
            Da = GB_LOGW + C + logD
            ma = float(Da.max())
            age = math.exp(ma + math.log(float(np.exp(Da - ma).sum())) - tot)
        else:
            age = 0.0
        out[t] = (tot + lq - ln, m + math.log(max(float(pr[up].sum()), 1e-300)) + lq - ln,
                  m + math.log(max(float(pr[~up].sum()), 1e-300)) + lq - ln, math.log1p(age),
                  float(pr @ _gl) / float(pr.sum()))
    return out


if os.path.exists(GB_PATH):
    GBX = np.load(GB_PATH)["X"]
    print(f"[gbocpd] loaded {GB_PATH}")
else:
    t0 = time.time()
    GBX = np.zeros((y_all.size, 5), dtype=np.float32)
    for k in range(_GU.size):
        rows = _order[_starts[k]:_ends[k]]
        h = gru_history(k)
        mu_h, sd_h = (float(h.mean()), float(h.std()) + 1e-6) if h.size >= 20 else (0.0, 1.0)
        GBX[rows] = _gbocpd_series((GS["rows_g"][rows] - mu_h) / sd_h)
    GBX = np.nan_to_num(GBX, nan=0.0, posinf=0.0, neginf=0.0)
    np.savez(GB_PATH, X=GBX, names=np.array(GB_NAMES))
    print(f"[gbocpd] computed for {_GU.size:,} series | {(time.time() - t0) / 60:.1f} min")

print("  univariate TS-AUC: " + ", ".join(f"{n} {max(a, 1 - a):.4f}" for n, a in
      ((n, ts_auc(y_all, t_all, GBX[:, j].astype(np.float64))) for j, n in enumerate(GB_NAMES))))
if "dbocpd_logodds" in BANK_NAMES:
    print(f"  for comparison, dbocpd_logodds (on e): {ts_auc(y_all, t_all, base_column('dbocpd_logodds').astype(np.float64)):.4f}")
_gb = run_cv(BASE_LABEL)
_gc = run_cv(f"{BASE_LABEL}+gbocpd", add=[(GBX, GB_NAMES)])
_c = compare(_gc, _gb)
print(f"\n=== dbocpd ON THE GRU'S SURPRISE STREAM vs {BASE_LABEL}")
for k_, p_ in _c.items():
    print(f"  {k_:9s}: {p_['delta']:+.5f} | se {p_['se']:.5f} | {p_['sigma']:+.1f} sigma | better {p_['better']}/{p_['n']}")
print("  VERDICT:", "ADOPT" if add_ok(_gc, _gb) else "REJECT")

[compat] nullmodel: wrote cache/new_nullmodel.npz (5036517, 17) (engine version 1)
[compat] dbocpd: wrote cache/new_dbocpd.npz (5036517, 5) (engine version 1)
[compat] now re-run the bank cell so the base picks these up
loaded crunch tools for module: <module '__main__'>

cli version: 12.0.3
available ram: 16.00 gb
available cpu: 10 core
----
data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_train.parquet (218514418 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/X_test.reduced.parquet (2587435 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/234/y_train.parquet (8356193 bytes)
data/y_train.parquet: already exists, file length match